In [ ]:
!/venv/main/bin/pip install numpy pandas transformers datasets accelerate scikit-learn -q

In [8]:
import torch
print(torch.__file__)

/venv/main/lib/python3.12/site-packages/torch/__init__.py


In [9]:
!/venv/main/bin/pip show torch

Name: torch
Version: 2.11.0
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License: BSD-3-Clause
Location: /venv/main/lib/python3.12/site-packages
Requires: cuda-bindings, cuda-toolkit, filelock, fsspec, jinja2, networkx, nvidia-cudnn-cu13, nvidia-cusparselt-cu13, nvidia-nccl-cu13, nvidia-nvshmem-cu13, setuptools, sympy, triton, typing-extensions
Required-by: accelerate, torchvision


In [11]:
!/venv/main/bin/pip install torch==2.7.0+cu128 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128 -q

In [1]:
import torch
print(torch.cuda.is_available())

True


In [3]:
import sys
print(sys.executable)

/venv/main/bin/python


In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

#import os
#os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import numpy as np# linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)



# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory



# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [5]:
import zipfile

with zipfile.ZipFile('/workspace/WELFake_Dataset.csv.zip', 'r') as z:
    z.extractall('/workspace/')

In [6]:
print(z.namelist())

['WELFake_Dataset.csv']


In [7]:
import os
os.path.exists('/workspace/WELFake_Dataset.csv')

True

# Loading & Cleaning Dataset

In [8]:
df = pd.read_csv('/workspace/WELFake_Dataset.csv')

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 72134 entries, 0 to 72133
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Unnamed: 0  72134 non-null  int64
 1   title       71576 non-null  str  
 2   text        72095 non-null  str  
 3   label       72134 non-null  int64
dtypes: int64(2), str(2)
memory usage: 235.0 MB


In [10]:
df.isnull().sum()

Unnamed: 0      0
title         558
text           39
label           0
dtype: int64

In [11]:
df = df[["title", "text", "label"]].dropna()

In [12]:
df.head()

,title,text,label
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1
5,About Time! Christian Group Sues Amazon and SP...,All we can say on this one is it s about time ...,1


In [13]:
df["text"] = df["title"].fillna("") + " " + df["text"].fillna("")
df = df[["text", "label"]].rename(columns={"label": "labels"})

In [14]:
df.head()

,text,labels
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,1
2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,1
3,"Bobby Jindal, raised Hindu, uses story of Chri...",0
4,SATAN 2: Russia unvelis an image of its terrif...,1
5,About Time! Christian Group Sues Amazon and SP...,1


In [15]:
df['labels'].value_counts()

labels
1    36509
0    35028
Name: count, dtype: int64

In [16]:
before = len(df)
df = df.drop_duplicates(subset=['text'])
after = len(df)

print("Before:", before)
print("After:", after)
print("Removed:", before - after)

Before: 71537
After: 63121
Removed: 8416


# Import Model

In [17]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer, TrainingArguments,
    DataCollatorWithPadding
)

In [18]:
from sklearn.model_selection import train_test_split
import torch
from datasets import Dataset

In [19]:

# First split: train (80%) and temp (20%)
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["labels"]   # important
)

# Second split: validation (10%) and test (10%)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["labels"]
)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Train: 50496
Val: 6312
Test: 6313


In [20]:
train_texts = set(train_df['text'])
val_texts = set(val_df['text'])
test_texts = set(test_df['text'])

print("Train ∩ Val:", len(train_texts & val_texts))
print("Train ∩ Test:", len(train_texts & test_texts))
print("Val ∩ Test:", len(val_texts & test_texts))

Train ∩ Val: 0
Train ∩ Test: 0
Val ∩ Test: 0


In [21]:
len(val_df)

6312

# Tokenization

In [22]:
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding=False,
        max_length=256
    )

train_ds = Dataset.from_pandas(train_df).map(tokenize, batched=True)
val_ds = Dataset.from_pandas(val_df).map(tokenize, batched=True)
test_ds = Dataset.from_pandas(test_df).map(tokenize, batched=True)

train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/50496 [00:00<?, ? examples/s]

Map:   0%|          | 0/6312 [00:00<?, ? examples/s]

Map:   0%|          | 0/6313 [00:00<?, ? examples/s]

# Model

In [23]:
model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2,
    label2id={"fake": 0, "real": 1},
    id2label={0: "fake", 1: "real"}
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [24]:
model

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [25]:
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

# Metrics

In [26]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)
    }

# Training Arguments

In [27]:



args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=False,
    metric_for_best_model="f1",
    fp16=True,
)

In [28]:
import torch
print(torch.cuda.is_available())
print(torch.version.cuda)

True
12.8


In [29]:
!nvidia-smi

Thu Apr 23 18:53:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.211.01             Driver Version: 570.211.01     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5070 Ti     On  |   00000000:04:00.0 Off |                  N/A |
|  0%   36C    P8             20W /  300W |      18MiB /  16303MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Trainer

In [30]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer),
)

# Train

In [31]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.015103,0.009445,0.997782,0.997528
2,0.006406,0.011332,0.997940,0.997709
3,0.000404,0.007410,0.999049,0.998942


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4734, training_loss=0.022271301734281282, metrics={'train_runtime': 558.81, 'train_samples_per_second': 271.09, 'train_steps_per_second': 8.472, 'total_flos': 1.992908377718784e+16, 'train_loss': 0.022271301734281282, 'epoch': 3.0})

In [32]:
model.save_pretrained("/workspace/roberta-fakedetect")
tokenizer.save_pretrained("/workspace/roberta-fakedetect")
print("Model saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved!


# Evaluation

In [33]:
print(train_ds.features)
print(train_ds[0]['labels'])
print(type(train_ds[0]['labels']))

{'text': Value('large_string'), 'labels': Value('int64'), '__index_level_0__': Value('int64'), 'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8'))}
tensor(0)
<class 'torch.Tensor'>


In [36]:
print(val_ds[0])

{'labels': tensor(1), 'input_ids': tensor([    0,  1585,   384,    17,    27, 16626, 44740,   293, 25657, 42338,
        18990, 21617,  1869,   374,  2588,  2461, 40495,   126,    85,    17,
           27,    29, 23743,   468,  1848,  2011,  2284,  1530,     9,  1144,
         1672,    82,    15,     5,   235,     6,  1585,   384, 24310,    34,
         7546,    11,    15,     5,  2588,  2461, 12998,    19,    65,     9,
            5,  2373,   383,    47,    64,   224,    59,   143,  2862,  1094,
            4,    20,   320,  2063,   491,  1482,    16,    41, 20137,   132,
         1187,  8352,  1437,  2378,     6,  1437,  3099,    37,  2046,    14,
         5013,    32,    55,   505,    87,    82,   579,  1074,     6,     8,
           37,  9180,   201,    14,    37,  2653,    14,   169,    11,    41,
         3668, 21096,  5059,   618,   452,     4,    20, 15090,     8,    63,
         2732,   236,  1365,   899,     7,  2398,     6,   150,     5,   314,
         1072,   106,  4968, 

In [34]:
from sklearn.metrics import classification_report
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding
import torch

model.eval()
all_preds = []
all_labels = []

collator = DataCollatorWithPadding(tokenizer)
dataloader = DataLoader(test_ds, batch_size=32, collate_fn=collator)

for batch in dataloader:
    with torch.no_grad():
        outputs = model(
            input_ids=batch['input_ids'].to('cuda'),
            attention_mask=batch['attention_mask'].to('cuda')
        )
    preds = torch.argmax(outputs.logits, dim=1).cpu().tolist()
    all_preds.extend(preds)
    all_labels.extend(batch['labels'].tolist())

print(classification_report(all_labels, all_preds, target_names=['fake', 'real']))

              precision    recall  f1-score   support

        fake       1.00      1.00      1.00      3480
        real       1.00      1.00      1.00      2833

    accuracy                           1.00      6313
   macro avg       1.00      1.00      1.00      6313
weighted avg       1.00      1.00      1.00      6313



In [35]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.000404,0.007410,3,0.999049,0.998942


{'eval_loss': 0.007409823592752218,
 'eval_accuracy': 0.9990494296577946,
 'eval_f1': 0.9989417989417989}

In [ ]:
test_results = trainer.evaluate(test_ds)
print(test_results)

In [37]:
# Check for duplicate texts between train and test
train_texts = set(train_df['text'].tolist())
test_texts = set(test_df['text'].tolist())

overlap = train_texts.intersection(test_texts)
print(f"Overlapping samples: {len(overlap)}")

Overlapping samples: 0


In [39]:
from datasets import load_dataset
gonzalo = load_dataset("GonzaloA/fake_news")

Repo card metadata block was not found. Setting CardData to empty.


In [40]:
print(gonzalo)
print(gonzalo['test'].features)

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'title', 'text', 'label'],
        num_rows: 24353
    })
    validation: Dataset({
        features: ['Unnamed: 0', 'title', 'text', 'label'],
        num_rows: 8117
    })
    test: Dataset({
        features: ['Unnamed: 0', 'title', 'text', 'label'],
        num_rows: 8117
    })
})
{'Unnamed: 0': Value('int64'), 'title': Value('string'), 'text': Value('string'), 'label': Value('int64')}


In [41]:
import pandas as pd
gonzalo_test_df = gonzalo['test'].to_pandas()
print(gonzalo_test_df['label'].value_counts())

label
1    4335
0    3782
Name: count, dtype: int64


In [42]:
gonzalo_test_df['text'] = gonzalo_test_df['title'].fillna('') + ' ' + gonzalo_test_df['text'].fillna('')
gonzalo_test_df = gonzalo_test_df[['text', 'label']]
print(gonzalo_test_df.shape)
print(gonzalo_test_df['label'].value_counts())

(8117, 2)
label
1    4335
0    3782
Name: count, dtype: int64


In [43]:
from datasets import Dataset

gonzalo_ds = Dataset.from_pandas(gonzalo_test_df)

def tokenize_gonzalo(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding=False,
        max_length=256
    )

gonzalo_ds = gonzalo_ds.map(tokenize_gonzalo, batched=True)
gonzalo_ds = gonzalo_ds.rename_column("label", "labels")
gonzalo_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# Run evaluation
model.eval()
all_preds = []
all_labels = []

collator = DataCollatorWithPadding(tokenizer)
dataloader = DataLoader(gonzalo_ds, batch_size=32, collate_fn=collator)

for batch in dataloader:
    with torch.no_grad():
        outputs = model(
            input_ids=batch['input_ids'].to('cuda'),
            attention_mask=batch['attention_mask'].to('cuda')
        )
    preds = torch.argmax(outputs.logits, dim=1).cpu().tolist()
    all_preds.extend(preds)
    all_labels.extend(batch['labels'].tolist())

print(classification_report(all_labels, all_preds, target_names=['fake', 'real']))

Map:   0%|          | 0/8117 [00:00<?, ? examples/s]

              precision    recall  f1-score   support

        fake       0.00      0.00      0.00      3782
        real       0.04      0.03      0.04      4335

    accuracy                           0.02      8117
   macro avg       0.02      0.02      0.02      8117
weighted avg       0.02      0.02      0.02      8117



In [44]:
print("WELFake mapping - 0 means:", "fake" if train_df[train_df['labels']==0]['text'].iloc[0] else "real")
print("\nGonzalo sample label 0:")
print(gonzalo_test_df[gonzalo_test_df['label']==0]['text'].iloc[0][:200])
print("\nGonzalo sample label 1:")
print(gonzalo_test_df[gonzalo_test_df['label']==1]['text'].iloc[0][:200])

WELFake mapping - 0 means: fake

Gonzalo sample label 0:
FORMER U.S. ATTORNEY: “It’s VERY Clear Intel Conspired to Frame Trump” (VIDEO) JOE DIGENOVA has been around D.C for decades and has seen it all. He probably didn t see his one coming. The incoming pre

Gonzalo sample label 1:
Nepal votes in final round of polls at the end of long democratic transition KATHMANDU (Reuters) - Nepalis began voting in the final round of parliamentary elections on Thursday, a key step to complet


In [45]:
print(train_df[train_df['labels']==0]['text'].iloc[0][:200])
print("---")
print(train_df[train_df['labels']==1]['text'].iloc[0][:200])

Trump tweets about Russia probe spark warnings from lawmakers WASHINGTON/NEW YORK (Reuters) - A series of tweets by U.S. President Donald Trump about the investigation into contacts between his 2016 c
---
ART OF WAR: What’s Behind Russia’s ‘Ides of March’ Military Drawdown in Syria?  Appear weak when you are strong, and strong when you are weak.    Sun Tzu, The Art of War21st Century Wire asks What s b


In [46]:
flipped_preds = [1 - p for p in all_preds]
print(classification_report(all_labels, flipped_preds, target_names=['fake', 'real']))

              precision    recall  f1-score   support

        fake       0.96      1.00      0.98      3782
        real       1.00      0.97      0.98      4335

    accuracy                           0.98      8117
   macro avg       0.98      0.98      0.98      8117
weighted avg       0.98      0.98      0.98      8117



In [47]:
# Take a clearly fake article from GonzaloA and see what the model predicts
sample = gonzalo_test_df[gonzalo_test_df['label']==0]['text'].iloc[0]
print("Article (should be FAKE):")
print(sample[:300])
print()

inputs = tokenizer(sample, return_tensors='pt', truncation=True, max_length=256).to('cuda')
with torch.no_grad():
    output = model(**inputs)
pred = torch.argmax(output.logits, dim=1).item()
print(f"Raw model prediction: {pred}")
print(f"Model says: {'fake' if pred == 1 else 'real'}")

Article (should be FAKE):
FORMER U.S. ATTORNEY: “It’s VERY Clear Intel Conspired to Frame Trump” (VIDEO) JOE DIGENOVA has been around D.C for decades and has seen it all. He probably didn t see his one coming. The incoming president  was set-up to be taken down. A soft coup is in the works and DiGenova has this to say about 

Raw model prediction: 1
Model says: fake


In [48]:
sample = gonzalo_test_df[gonzalo_test_df['label']==1]['text'].iloc[0]
print("Article (should be REAL):")
print(sample[:300])
print()

inputs = tokenizer(sample, return_tensors='pt', truncation=True, max_length=256).to('cuda')
with torch.no_grad():
    output = model(**inputs)
pred = torch.argmax(output.logits, dim=1).item()
print(f"Raw model prediction: {pred}")
print(f"Model says: {'fake' if pred == 1 else 'real'}")

Article (should be REAL):
Nepal votes in final round of polls at the end of long democratic transition KATHMANDU (Reuters) - Nepalis began voting in the final round of parliamentary elections on Thursday, a key step to complete a near decade-long democratic transition after the abolition of the centuries-old monarchy and the

Raw model prediction: 0
Model says: real


In [49]:
model.config.label2id = {"real": 0, "fake": 1}
model.config.id2label = {0: "real", 1: "fake"}
model.save_pretrained("/workspace/roberta-fakedetect")
tokenizer.save_pretrained("/workspace/roberta-fakedetect")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/workspace/roberta-fakedetect/tokenizer_config.json',
 '/workspace/roberta-fakedetect/tokenizer.json')

In [51]:
import pandas as pd

train_liar = pd.read_csv('/workspace/train_clean.csv')
val_liar = pd.read_csv('/workspace/val_clean.csv')
test_liar = pd.read_csv('/workspace/test_clean.csv')

print(test_liar.shape)
print(test_liar.columns.tolist())
print(test_liar.head())

(1267, 2)
['statement', 'label']
                                           statement      label
0  building a wall on the u.s.-mexico border will...       real
1  wisconsin is on pace to double the number of l...       fake
2  says john mccain has done nothing to help the ...       fake
3  suzanne bonamici supports a plan that will cut...  uncertain
4  when asked by a reporter whether hes at the ce...       fake


In [52]:
print(test_liar['label'].value_counts())

label
uncertain    477
real         449
fake         341
Name: count, dtype: int64


In [53]:
liar_binary = test_liar[test_liar['label'] != 'uncertain'].copy()
liar_binary['label'] = liar_binary['label'].map({'real': 0, 'fake': 1})
print(liar_binary.shape)
print(liar_binary['label'].value_counts())

(790, 2)
label
0    449
1    341
Name: count, dtype: int64


In [54]:
from datasets import Dataset

liar_ds = Dataset.from_pandas(liar_binary.rename(columns={'statement': 'text', 'label': 'labels'}))
liar_ds = liar_ds.map(lambda batch: tokenizer(batch['text'], truncation=True, padding=False, max_length=256), batched=True)
liar_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

model.eval()
all_preds = []
all_labels = []

dataloader = DataLoader(liar_ds, batch_size=32, collate_fn=DataCollatorWithPadding(tokenizer))

for batch in dataloader:
    with torch.no_grad():
        outputs = model(
            input_ids=batch['input_ids'].to('cuda'),
            attention_mask=batch['attention_mask'].to('cuda')
        )
    preds = torch.argmax(outputs.logits, dim=1).cpu().tolist()
    all_preds.extend(preds)
    all_labels.extend(batch['labels'].tolist())

print(classification_report(all_labels, all_preds, target_names=['real', 'fake']))

Map:   0%|          | 0/790 [00:00<?, ? examples/s]

              precision    recall  f1-score   support

        real       0.57      0.90      0.70       449
        fake       0.48      0.12      0.20       341

    accuracy                           0.56       790
   macro avg       0.53      0.51      0.45       790
weighted avg       0.53      0.56      0.48       790



In [57]:
!/venv/main/bin/pip install kagglehub -q

In [58]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("clmentbisaillon/fake-and-real-news-dataset")

print("Path to dataset files:", path)

100%|██████████| 41.0M/41.0M [00:02<00:00, 17.0MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/clmentbisaillon/fake-and-real-news-dataset/versions/1


In [59]:
import os
path = "/root/.cache/kagglehub/datasets/clmentbisaillon/fake-and-real-news-dataset/versions/1"
print(os.listdir(path))

['Fake.csv', 'True.csv']


In [60]:
fake_df = pd.read_csv(f"{path}/Fake.csv")
true_df = pd.read_csv(f"{path}/True.csv")

fake_df['label'] = 1
true_df['label'] = 0

isot_df = pd.concat([fake_df, true_df], ignore_index=True)
print(isot_df.shape)
print(isot_df.columns.tolist())
print(isot_df['label'].value_counts())

(44898, 5)
['title', 'text', 'subject', 'date', 'label']
label
1    23481
0    21417
Name: count, dtype: int64


In [61]:
isot_df['text'] = isot_df['title'].fillna('') + ' ' + isot_df['text'].fillna('')
isot_test = isot_df[['text', 'label']].sample(n=5000, random_state=42)  # sample 5000 for speed

isot_ds = Dataset.from_pandas(isot_test.rename(columns={'label': 'labels'}))
isot_ds = isot_ds.map(lambda batch: tokenizer(batch['text'], truncation=True, padding=False, max_length=256), batched=True)
isot_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

model.eval()
all_preds = []
all_labels = []

dataloader = DataLoader(isot_ds, batch_size=32, collate_fn=DataCollatorWithPadding(tokenizer))

for batch in dataloader:
    with torch.no_grad():
        outputs = model(
            input_ids=batch['input_ids'].to('cuda'),
            attention_mask=batch['attention_mask'].to('cuda')
        )
    preds = torch.argmax(outputs.logits, dim=1).cpu().tolist()
    all_preds.extend(preds)
    all_labels.extend(batch['labels'].tolist())

print(classification_report(all_labels, all_preds, target_names=['real', 'fake']))

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

              precision    recall  f1-score   support

        real       1.00      1.00      1.00      2350
        fake       1.00      1.00      1.00      2650

    accuracy                           1.00      5000
   macro avg       1.00      1.00      1.00      5000
weighted avg       1.00      1.00      1.00      5000



In [62]:
import shutil
shutil.make_archive('/workspace/roberta-fakedetect', 'zip', '/workspace/roberta-fakedetect')

'/workspace/roberta-fakedetect.zip'